# ADRec Training on Amazon Baby

Train and evaluate ADRec using the shared dataset loader and evaluation code. Run the sections in order.

Before running the notebook:

1. Attach the Kaggle input containing `cmpe257_kaggle_code.zip` and `dataset.pkl`.
2. Enable Internet access for dependency installation.
3. Select a GPU for full training. The notebook uses one GPU.

The setup accepts either the code ZIP or its extracted contents. Detailed configuration and checkpoint instructions are in `docs/KAGGLE_TRAINING.md`.


## 1. Load the project files

Locate the code and dataset under `/kaggle/input`. Extract or copy the code into `/kaggle/working/CMPE257_ML`, verify the source hashes, and add the project to the Python import path.

Leave the path variables as `None` for automatic discovery. When multiple matching inputs are attached, set `CODE_ZIP` or `CODE_DIR`, and `DATA_FILE`, to the required paths.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# None enables automatic discovery within the attached Kaggle inputs.
# Set explicit paths when multiple matching files are attached.
CODE_ZIP = None  # Path to the attached code ZIP
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATA_FILE = None  # Path to the attached Baby dataset
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

DATASET_PATH = attached_file('dataset.pkl', DATA_FILE)
REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
print('Dataset:', DATASET_PATH)


## 2. Install dependencies

Install the packages listed in `requirements-shared.txt`. The installation retains the existing CUDA-enabled PyTorch package when it satisfies the requirements. Complete this cell before importing the models.

If installation requests a kernel restart, restart the session and rerun the setup and installation cells.


In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)


## 3. Check the runtime

Import the shared data, training, and inference functions. Check the printed PyTorch version and GPU information. Full training requires `CUDA available: True`; smoke runs also support CPU execution.


In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Check model imports before starting training.
from sasrec import SASRec
from model import Att_Diffuse_model


## 4. Configure the run and validate the data

Set `RUN_MODE` to `smoke` for the initial pipeline check. This mode uses nine users, hidden size 16, and two epochs per stage. Smoke results measure pipeline execution only.

Set `RUN_MODE` to `full` for the complete dataset, hidden size 128, and up to 500 epochs with early stopping. Use a different `RUN_NAME` for each new experiment. Full runs use an evaluation batch size of 512.

Leave `RESUME_CHECKPOINT = None` for a new run. Set `FINALIZE = True` to evaluate the selected model on the test set after training. Set it to `False` for intermediate training sessions.

Load the dataset, verify its checksum, and print the user and interaction counts. The supplied train, validation, and test splits are retained.


In [ ]:
RUN_MODE = 'smoke'  # 'full' uses all 11,761 users, hidden size 128, up to 500 epochs
RUN_NAME = 'adrec_' + RUN_MODE + '_01'
FINALIZE = True
RESUME_CHECKPOINT = None  # Path to this model's latest.pt from a saved previous run
cfg = make_config('adrec', RUN_MODE)
# Full evaluation uses batches of 512; training batch size is configured separately.
RUN_DIR = WORK_ROOT / 'runs' / RUN_NAME
print(json.dumps(cfg, indent=2))
data = load_dataset(DATASET_PATH, cfg['dataset_sha256'])
print(dataset_summary(data))


## 5. Pretrain the item embeddings

Train the original `pretrain` model using next-item prediction and cross-entropy loss. This model uses the SASRec implementation without positional embeddings. Validation selects `pretrain/best.pt`; only its item embeddings are transferred to ADRec. The test split is not evaluated during pretraining.

Leave `PRETRAINED_CHECKPOINT = None` to train new embeddings. To load existing embeddings, set it to a compatible Baby checkpoint with 4,732 embedding rows and the selected hidden size. Confirm that its item-ID mapping matches the dataset. Full runs require 128-dimensional embeddings; smoke checkpoints use 16 dimensions.

Set `PRETRAIN_RESUME` to `pretrain/latest.pt` to resume this stage. Complete pretraining before starting ADRec. When `RESUME_CHECKPOINT` points to an ADRec checkpoint, this cell skips pretraining.


In [ ]:
PRETRAINED_CHECKPOINT = None  # Optional path to existing matching Baby embeddings
PRETRAIN_RESUME = None       # Optional path to pretrain/latest.pt
pretrain_cfg = make_config('pretrain', RUN_MODE)
if RESUME_CHECKPOINT is None and PRETRAINED_CHECKPOINT is None:
    pretrain_result = train(pretrain_cfg, DATASET_PATH, RUN_DIR / 'pretrain',
                            resume=PRETRAIN_RESUME, finalize=False)
    PRETRAINED_CHECKPOINT = RUN_DIR / 'pretrain' / 'best.pt'
print('Embedding checkpoint:', PRETRAINED_CHECKPOINT)


## 6. Train ADRec

Load the pretrained item embeddings and train the ADRec backbone. In full mode, embeddings remain frozen for epochs 0–4 and become trainable at epoch 5. Smoke mode uses one frozen epoch followed by one fine-tuning epoch. The optimizer and scheduler continue across this transition.

Training uses recommendation and diffusion losses. Validation selects the saved model, and `FINALIZE = True` enables final test evaluation. Each completed epoch saves a resume checkpoint. To continue an existing run, provide its resume checkpoint; a new run requires a new output directory.


In [ ]:
result = train(cfg, DATASET_PATH, RUN_DIR / 'adrec',
               pretrained_checkpoint=PRETRAINED_CHECKPOINT,
               resume=RESUME_CHECKPOINT, finalize=FINALIZE)
MODEL_DIR = RUN_DIR / 'adrec'


## 7. Inspect the results

Display the run summary, selected epoch, final test metrics, and saved filenames. This cell reads the completed run and does not train the model.

HR and NDCG are reported as percentages at cutoffs 5, 10, and 20. Evaluation retains the upstream candidate columns and averages batch means. Use validation for model selection and test results for final reporting. `test: null` indicates that final test evaluation was disabled.


In [ ]:
print(json.dumps(result, indent=2))
print('Saved files:', [p.name for p in MODEL_DIR.iterdir()])


## 8. Load the checkpoint and generate recommendations

Load `best.pt` and generate ten recommendations from a user's visible validation history. The held-out target is excluded from the prediction input. The seed controls the random state for inference.

The output contains product IDs because the dataset does not include product titles or images. Padding ID 0 is excluded from this example's recommendations; the benchmark evaluation retains its original candidate rules. Previously seen products remain eligible.


In [ ]:
model, saved_cfg = load_for_inference(MODEL_DIR / 'best.pt')
example = SequenceDataset(data, 'val', saved_cfg['max_len'])[0]
seed_everything(saved_cfg['seed'])
scores = predict_scores(model, [example['input_ids']]).cpu()
scores[:, 0] = -torch.inf  # Exclude padding from the inference example
recommended_ids = torch.topk(scores, k=10, dim=-1).indices[0].tolist()
print('Visible history:', [i for i in example['input_ids'] if i])
print('Recommended product IDs:', recommended_ids)


## 9. Save the output files

Download the complete run directory at `/kaggle/working/runs/<RUN_NAME>/` before ending the session. Retain both the `pretrain/` and `adrec/` folders.

| File | Contents |
|---|---|
| `best.pt` | Validation-selected weights and settings for inference |
| `latest.pt` | Training state for resuming from the last completed epoch |
| `config.json` | Training settings and dataset checksum |
| `environment.json` | Runtime information and source version |
| `requirements-lock.txt` | Installed package versions |
| `history.json` | Training losses and validation results |
| `results.json` | Run summary and final test results |

Use Kaggle's output file browser to download the files. Store model weights under the local project's ignored `checkpoints/` directory. Kaggle **Save & Run All** starts a separate execution from the first cell; it does not continue the current interactive execution.


## 10. Resume a saved run

1. Attach the saved `latest.pt` as a Kaggle input.
2. Restore the original run mode and training settings.
3. Set `RESUME_CHECKPOINT` to the attached checkpoint path and set a new `RUN_NAME` for the output directory.
4. Run the notebook from the setup section. Training resumes from the last completed epoch.

For embedding pretraining, set `PRETRAIN_RESUME` instead. Resuming ADRec skips embedding pretraining. Only the total `epochs` setting can increase on resume. Other configuration changes are rejected. The optimizer, scheduler, and random states are restored. Numerical results can differ across hardware and library versions. An early-stopped run remains stopped.

Set `FINALIZE = False` for intermediate sessions and `True` for the final evaluation. To evaluate a completed run without further training, resume with its original epoch limit and `FINALIZE = True`.
